In [ ]:
import os
import sys
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate

# 1. Enrutamiento del entorno
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.append(root_path)

from src.utils.config import get_llm

print(">>> INICIANDO CONFIGURACIÓN DEL ASISTENTE Q&A <<<")

# 2. Configuración de la Memoria Conversacional
# Retiene el contexto de las preguntas anteriores del usuario
memory = ConversationBufferMemory(
    memory_key="chat_history", 
    return_messages=True,
    input_key="question",
    output_key="answer"
)

# 3. Diseño del Prompt (Personalidad de Consultor Senior)
prompt_qa = PromptTemplate(
    input_variables=["context", "question", "chat_history"],
    template=(
        "Actúa como un Consultor Senior experto de ConsulTech Analytics.\n"
        "Tu objetivo es responder a la pregunta del usuario utilizando estrictamente la información "
        "encontrada en el 'Contexto del documento'. No inventes datos. Si la respuesta no está en el contexto, "
        "indica amablemente que el documento no contiene esa información.\n\n"
        "Historial de la conversación:\n{chat_history}\n\n"
        "Contexto del documento:\n{context}\n\n"
        "Pregunta: {question}\n\n"
        "Respuesta del Consultor Senior:"
    )
)

llm = get_llm(temperature=0.2) # Temperatura baja para mantener un tono formal y preciso

print("✅ Memoria conversacional y Prompt del Consultor Senior configurados.")

In [ ]:
# Extracto de cómo tu compañero debe estar consumiendo tu código en el Notebook:
from src.agent.qa_system import setup_qa_chain

# 1. Inicializamos tu motor
qa_chain = setup_qa_chain()
historial_chat = [] # Memoria de la conversación

# 2. Hacemos una pregunta de prueba
pregunta = "¿Cuáles son las conclusiones principales del documento?"

# 3. Ejecución
respuesta = qa_chain.invoke({"question": pregunta, "chat_history": historial_chat})

print("🤖 Asistente:", respuesta["answer"])
print("\n📑 Fuentes Citadas:")
for doc in respuesta["source_documents"]:
    print(f"- Fragmento extraído de: {doc.metadata.get('source', 'Desconocido')}")
    
# 4. Actualizar memoria (Trabajo de tu compañero)
historial_chat.append((pregunta, respuesta["answer"]))